In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import glob
import xarray as xr
import xbudget
import regionate
import xwmt
import xwmb
import xgcm
import cartopy.crs as ccrs
import xesmf as xe
import CM4Xutils #needed to run pip install nc-time-axis
from regionate import MaskRegions, GriddedRegion
import sys
sys.path.insert(0, '/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/src')
from src import *

In [64]:
rename_geocoords = lambda ds: ds.rename({"lat":"old_lat", "lon":"old_lon"}).rename({"geolat":"lat", "geolon":"lon"})

def setup_regridder(ds_locs, ds_fname):
    ds_tmp = read_tracer_and_zos_from_budget(ds_fname)

    ds_tmp_coords = ds_tmp[["geolat", "geolon"]]
    ds_tmp_coords = rename_geocoords(ds_tmp_coords).compute()
    # ds_tmp_coords = ds_tmp_coords.compute()

    regridder = xe.Regridder(ds_tmp_coords, ds_locs, "bilinear", locstream_out=True)

    return regridder

def apply_regridder(regridder, ds):
    ds_sampled = regridder(ds.reset_coords(["geolon", "geolat", "wet"], drop = False))

    ds_sampled = ds_sampled.drop_vars(['geolat_c', 'geolon_c', "xq", "yq"])
    ds_sampled = ds_sampled.set_coords(["wet", "geolon", "geolat"])
    for v in ds_sampled.data_vars:
        ds_sampled[v].attrs.update(ds_tmp[v].attrs)
    
    return ds_sampled

def remap_samples_to_depth(ds_sampled):
    ds_remap = remap_sigma_to_depth(ds_sampled.where(ds_sampled.wet > 0))
    return ds_remap

In [5]:
from dask_jobqueue import SLURMCluster  # setup dask cluster 
from dask.distributed import Client

log_directory="/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/logs"

cluster = SLURMCluster(
    cores=36,
    processes=1,
    memory='190GB',
    walltime='03:00:00',
    queue='compute',
    interface='ib0', 
log_directory = log_directory)
print(cluster.job_script())
cluster.scale(jobs=4)

client = Client(cluster)
client

#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /vortexfs1/home/anthony.meza/scratch/CM4X/CM4XTransientTracers/WaterMassBudgets/logs/dask-worker-%J.err
#SBATCH -o /vortexfs1/home/anthony.meza/scratch/CM4X/CM4XTransientTracers/WaterMassBudgets/logs/dask-worker-%J.out
#SBATCH -p compute
#SBATCH -n 1
#SBATCH --cpus-per-task=36
#SBATCH --mem=177G
#SBATCH -t 03:00:00

/vortexfs1/home/anthony.meza/miniforge3/envs/cm4x_analysis/bin/python -m distributed.cli.dask_worker tcp://172.16.3.85:38819 --name dummy-name --nthreads 36 --memory-limit 176.95GiB --nanny --death-timeout 60 --interface ib0



Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.16.3.85:38819,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [250]:
expt_datafiles = collect_tracer_files(model="CM4Xp125")
del expt_datafiles["spinup"]
expt_datafiles.keys()

dict_keys(['forced', 'control'])

In [251]:
ds_locs = xr.Dataset(
    {
        "lat": (["locations"], [-5., -77.]),
        "lon": (["locations"], [-177., -177.]),
    }
)

In [252]:
regridder = setup_regridder(ds_locs, expt_datafiles["forced"][0])

In [275]:
ds_samp_ts_dict = {}
for key in expt_datafiles.keys():
    datafiles = expt_datafiles[key]
    ds_samp_ts = []
    for (t, file) in enumerate(datafiles[17:]): 
        if t % 5 == 0:
            print(key, ":", file)
        ds_tmp = read_tracer_and_zos_from_budget(file)
        ds_tmp["rho2"] = ds_tmp.sigma2_l * xr.where(ds_tmp.thkcello.fillna(0.0) > 0.0, 1, 0) 
        ds_tmp["rho2"] = ds_tmp["rho2"].where(ds_tmp["rho2"] > 0)
        ds_tmp["rho2"].attrs.update({
        "cell_methods": ds_tmp["thetao"].attrs["cell_methods"],
        "standard_name": "sea_water_potential_density",
        "units": "kg / m^3",
        "description": "Potential Density references to 2000 dbar"
        })
        
        ds_sampled = apply_regridder(regridder, ds_tmp).compute()
        ds_samp_ts += [ds_sampled.copy()]
    ds_samp_ts_dict[key] = xr.concat(ds_samp_ts, dim = "time", combine_attrs = "identical").sortby("time")

    # ds_sampled_z = remap_sigma_columns_to_depth(ds_sampled).compute()

ds_samp_ts_list = [
    ds_samp_ts_dict[dkey].expand_dims(exp=[dkey]) 
    for dkey in ds_samp_ts_dict.keys()
]

combined_ds_samp = xr.concat(ds_samp_ts_list, dim="exp", combine_attrs = "identical")

savedir = "/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/"
savename = savedir + f"Column_Tracer_Samples_On_Sigma_Example.nc"
print(f"Saving {key} to", ": ", savename)
combined_ds_samp.to_netcdf(savename)

forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_historical_tracers_sigma2_1935-1939.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_historical_tracers_sigma2_1960-1964.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_historical_tracers_sigma2_1985-1989.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_historical_tracers_sigma2_2010-2014.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_ssp585_tracers_sigma2_2035-2039.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_ssp585_tracers_sigma2_2060-2064.zarr
forced : /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/tracers_sigma2_1p5/CM4Xp125_ssp585_tracers_sigma2_2085-2089.zarr
control : /vortexf